In [66]:
!pip install -q pymupdf sentence-transformers openai tiktoken

In [67]:
GROQ_API_KEY = "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" #we will use this api in env variable to secure it.

In [68]:
from google.colab import files

uploaded = files.upload()
PDF_PATH = next(iter(uploaded))

print("Uploaded file:", PDF_PATH)

Saving Test PDF (Hard).pdf to Test PDF (Hard) (2).pdf
Uploaded file: Test PDF (Hard) (2).pdf


In [69]:
import re
import fitz
import unicodedata

from dataclasses import dataclass
from typing import List, Dict, Any

from sentence_transformers import SentenceTransformer, util
from openai import OpenAI

In [70]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GROQ_MODEL = "llama-3.1-8b-instant"

CHUNK_SIZE_WORDS = 180
CHUNK_OVERLAP_WORDS = 70
TOP_K = 6
MAX_CONTEXT_CHARS = 18000

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

embed_model = SentenceTransformer(EMBED_MODEL)

print("Ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready.


In [71]:
def is_mostly_noise(line: str) -> bool:
    s = line.strip()
    if not s:
        return True

    total = len(s)
    letters = sum(ch.isalpha() for ch in s)
    digits = sum(ch.isdigit() for ch in s)
    spaces = sum(ch.isspace() for ch in s)
    punct = total - letters - digits - spaces

    # line is mostly punctuation/symbols and has very little text
    if total >= 8 and punct / total > 0.6 and letters / total < 0.2:
        return True

    # low-variety garbage like )))))) or $$$$$$
    unique_chars = len(set(s))
    if total >= 8 and unique_chars <= 3 and letters == 0 and digits == 0:
        return True

    return False


def normalize_pdf_text(text: str) -> str:
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    cleaned_lines = []

    for line in text.split("\n"):
        line = line.strip()

        if not line:
            continue

        line = re.sub(r"\s+", " ", line)
        line = re.sub(r'([^\w\s])\1{4,}', ' ', line)
        line = re.sub(r'[^\w\s]{5,}', ' ', line)
        line = re.sub(r"\s+", " ", line).strip()

        if not line:
            continue

        if is_mostly_noise(line):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    # join wrapped lines into normal sentences
    text = re.sub(r'(?<![.!?:])\n(?!\n)', ' ', text)

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n\n', text)

    return text.strip()

In [72]:
def extract_text_from_pdf(pdf_path: str) -> List[Dict[str, Any]]:
    doc = fitz.open(pdf_path)
    pages = []

    for page_num, page in enumerate(doc):
        raw_text = page.get_text("text")
        clean_text = normalize_pdf_text(raw_text)

        if clean_text:
            pages.append({
                "page": page_num + 1,
                "text": clean_text
            })

    doc.close()
    return pages

In [73]:
pages = extract_text_from_pdf(PDF_PATH)

for p in pages:
    print(f"\n--- PAGE {p['page']} ---\n")
    print(p["text"][:2000])


--- PAGE 1 ---

Bangladesh(_ ( _ && ( _ ( _ ( _. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political center but also the economic and cultural heart of the nation. It is one of the most densely populated cities in the world and plays a major role in the country’s development.
One of the most important natural features of Bangladesh is the (_ () _ ( _ $$ ( _ ( _ ( _.
Padma River, which is often referred to as the lifeline of the country. This river is crucial for transportation, agriculture, fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile land that makes Bangladesh suitable for farming.
Bangladesh gained its independence in 1971. This historic event is remembered every year as Independence Day and is a source of great pride for the people of Bangladesh. The struggle for independence played a key role 

In [74]:
@dataclass
class Chunk:
    text: str
    page: int
    chunk_id: int


def chunk_text(text: str, page: int, chunk_size_words: int, chunk_overlap_words: int) -> List[Chunk]:
    words = text.split()
    chunks = []
    start = 0
    chunk_id = 0

    while start < len(words):
        end = min(start + chunk_size_words, len(words))
        chunk_words = words[start:end]
        chunk_str = " ".join(chunk_words).strip()

        if chunk_str:
            chunks.append(
                Chunk(
                    text=chunk_str,
                    page=page,
                    chunk_id=chunk_id
                )
            )

        if end == len(words):
            break

        start += chunk_size_words - chunk_overlap_words
        chunk_id += 1

    return chunks

In [75]:
all_chunks = []

for page_data in pages:
    page_chunks = chunk_text(
        text=page_data["text"],
        page=page_data["page"],
        chunk_size_words=CHUNK_SIZE_WORDS,
        chunk_overlap_words=CHUNK_OVERLAP_WORDS
    )
    all_chunks.extend(page_chunks)

if not all_chunks:
    raise ValueError("No readable text was extracted from the PDF.")

print("Total chunks:", len(all_chunks))

chunk_texts = [c.text for c in all_chunks]

chunk_embeddings = embed_model.encode(
    chunk_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Index built.")

Total chunks: 4


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Index built.


In [76]:
def retrieve_chunks(question: str, top_k: int = TOP_K):
    query_embedding = embed_model.encode(question, convert_to_tensor=True)

    hits = util.semantic_search(
        query_embedding,
        chunk_embeddings,
        top_k=top_k
    )[0]

    results = []
    for hit in hits:
        chunk = all_chunks[hit["corpus_id"]]
        results.append({
            "text": chunk.text,
            "page": chunk.page,
            "chunk_id": chunk.chunk_id,
            "retrieval_score": float(hit["score"])
        })

    return sorted(results, key=lambda x: x["retrieval_score"], reverse=True)

In [77]:
def build_context(retrieved_chunks, max_chars: int = MAX_CONTEXT_CHARS) -> str:
    parts = []
    total_chars = 0

    for i, ch in enumerate(retrieved_chunks, start=1):
        block = (
            f"Source {i}\n"
            f"Page: {ch['page']}\n"
            f"Content: {ch['text']}\n"
        )

        if total_chars + len(block) > max_chars:
            break

        parts.append(block)
        total_chars += len(block)

    return "\n\n".join(parts)

In [78]:
def ask_groq_with_context(question: str, context: str) -> str:
    system_prompt = """You answer using ONLY the provided PDF context.

Rules:
1. Answer only from the provided PDF context.
2. If the answer is not explicitly supported by the context, return exactly:
I don't have that information.
3. Do not use outside knowledge.
4. Do not invent facts.
5. Give one clear, natural answer to the user's question.
6. Do not turn the response into multiple questions and answers.
7. Do not produce a numbered list unless the user explicitly asked for a list.
8. For fact questions, answer in 1-2 sentences.
9. For broader questions, give a short paragraph summary only if that summary is directly supported by the provided context.
"""

    user_prompt = f"""Question:
{question}

PDF Context:
{context}
"""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    return response.choices[0].message.content.strip()

In [79]:
def answer_from_pdf(question: str, top_k: int = TOP_K):
    retrieved = retrieve_chunks(question, top_k=top_k)
    context = build_context(retrieved)
    answer = ask_groq_with_context(question, context)

    return {
        "answer": answer,
        "retrieved_chunks": retrieved,
        "context": context
    }

In [80]:
question = "Which river is considered the lifeline of Bangladesh?"
retrieved = retrieve_chunks(question)

for i, ch in enumerate(retrieved, 1):
    print(f"\n--- Chunk {i} | Page {ch['page']} | Score {ch['retrieval_score']:.4f} ---")
    print(ch["text"][:700])


--- Chunk 1 | Page 1 | Score 0.6757 ---
Bangladesh(_ ( _ && ( _ ( _ ( _. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political center but also the economic and cultural heart of the nation. It is one of the most densely populated cities in the world and plays a major role in the country’s development. One of the most important natural features of Bangladesh is the (_ () _ ( _ $$ ( _ ( _ ( _. Padma River, which is often referred to as the lifeline of the country. This river is crucial for transportation, agriculture, fishing, and daily life. Many people depend on it for their livelihoods, and it suppo

--- Chunk 2 | Page 1 | Score 0.5619 ---
This river is crucial for transportation, agriculture, fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile land that makes Bangladesh suitable for farming. Bangladesh g

In [81]:
question = "What is the capital city of Bangladesh?"
result = answer_from_pdf(question)

print("ANSWER:\n")
print(result["answer"])

ANSWER:

The capital city of Bangladesh is Dhaka.


In [82]:
while True:
    q = input("\nAsk a question (or type exit): ").strip()
    if q.lower() == "exit":
        break

    result = answer_from_pdf(q)

    print("\nANSWER:\n")
    print(result["answer"])


Ask a question (or type exit): What is the capital city of Bangladesh?

ANSWER:

The capital city of Bangladesh is Dhaka.

Ask a question (or type exit): Which river is considered the lifeline of Bangladesh?

ANSWER:

The Padma River is considered the lifeline of Bangladesh.

Ask a question (or type exit): In which year did Bangladesh gain independence?

ANSWER:

Bangladesh gained its independence in 1971.

Ask a question (or type exit): What is the official language of Bangladesh?

ANSWER:

The official language of Bangladesh is Bengali, also known as Bangla.

Ask a question (or type exit): What is the name of the world’s largest mangrove forest located in Bangladesh?

ANSWER:

The name of the world’s largest mangrove forest located in Bangladesh is the Sundarbans.

Ask a question (or type exit): Which currency is used in Bangladesh? What is the national animal of Bangladesh?

ANSWER:

The currency used in Bangladesh is the Bangladeshi Taka. The national animal of Bangladesh is the R